# Dynex SDK - nBit Adder Native Gate Circuit Example

First we import the required packages:

In [1]:
from pennylane import numpy as np
import pennylane as qml
from dynex import DynexConfig, ComputeBackend, DynexCircuit

config = DynexConfig(compute_backend=ComputeBackend.QPU, qpu_model="apollo_rc1", use_notebook_output=True)

We define our circuit:

In [2]:
params = [28938284928182, 312722366482869645212131]  # two numbers to add


def Nqubits(a, b):
    mxVal = a + b
    return mxVal.bit_length()


wires = Nqubits(*params)


def Kfourier(k, wires):
    for j in range(len(wires)):
        qml.RZ(k * np.pi / (2 ** j), wires=wires[j])


def FullAdder(params, state=True):
    a, b = params
    wires = Nqubits(a, b)
    qml.BasisEmbedding(a, wires=range(wires))
    qml.QFT(wires=range(wires))
    Kfourier(b, range(wires))
    qml.adjoint(qml.QFT)(wires=range(wires))
    if state:
        return qml.state()
    else:
        return qml.sample()

We draw the circuit:

In [3]:
# draw circuit:
# too large to draw _ = qml.draw_mpl(FullAdder, style="black_white")(params)
print("Cant't draw circuit with", Nqubits(params[0], params[1]), "wires");

Cant't draw circuit with 79 wires


We execute and measure the circuit on the Dynex platform:

In [4]:
# Execute the circuit on Dynex:
dynex_circuit = DynexCircuit(config=config)
measure = dynex_circuit.execute(FullAdder, params, wires, method="measure",
                                num_reads=1, integration_steps=10)
print("Mesaure:", measure)

INFO: [DYNEX-APOLLO-RC1] Executing PennyLane quantum circuit
INFO: [DYNEX-APOLLO-RC1] Sampler initialised
INFO: [DYNEX-APOLLO-RC1] Apollo QPU chip: apollo_rc1
INFO: [DYNEX-APOLLO-RC1] Settings: num_reads=1, shots=1, annealing_time=10
INFO: [DYNEX-APOLLO-RC1] Submitting the job to Dynex.
INFO: [DYNEX-APOLLO-RC1] SUCCESS: Job created successfully (job_id=7415)
INFO: [DYNEX-APOLLO-RC1] feed_dict: {'cos_rz_0': 0.9484517002617644, 'cos_rz_1': 0.987028799038246, 'cos_rz_10': 0.960342759269009, 'cos_rz_11': -0.9900360496641042, 'cos_rz_12': 0.07058310823382546, 'cos_rz_13': 0.7316362170620812, 'cos_rz_14': -0.9304934758132593, 'cos_rz_15': -0.18642226823362693, 'cos_rz_16': 0.6378000202909895, 'cos_rz_17': -0.9049309421969693, 'cos_rz_18': -0.21802414751929503, 'cos_rz_19': 0.6252902735852786, 'cos_rz_2': -0.9967519247631895, 'cos_rz_20': 0.9014683226784174, 'cos_rz_21': -0.9750559785669788, 'cos_rz_22': 0.11167815684595883, 'cos_rz_23': 0.7455461611617213, 'cos_rz_24': 0.9342232498610066, 'c

Mesaure: [1 0 0 0 0 1 0 0 0 1 1 1 0 0 0 1 0 1 1 0 1 1 0 1 0 1 0 0 1 0 1 0 0 1 0 1 1
 0 0 0 1 1 0 0 0 0 1 1 0 0 1 0 0 0 0 1 0 1 1 0 0 0 1 1 0 0 0 0 1 1 0 1 0 0
 1 1 0 0 1]


In [5]:
bitStr = "".join(map(str, measure.astype(int)))
dynexResult = int(bitStr, 2)
print("Dynex Result:", dynexResult)
print("Expected Result:", sum(params))
isValidDynex = dynexResult == sum(params)
print("Is Dynex Result Valid?", isValidDynex)

Dynex Result: 312722366511807930140313
Expected Result: 312722366511807930140313
Is Dynex Result Valid? True
